# 02 — Feature Extraction v2

Notebook này chạy pha feature extraction theo `feature_extraction_standard_v2.md`.

- Input: `data/processed_v4_rgb248_r4_exact/manifest.csv`
- Feature families: `always-on`, `conditional CFA`, `research-only`
- Core rule: notebook chỉ orchestration; toàn bộ logic trích xuất nằm trong `src/feature_extraction`.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_extraction import (
    ALL_FEATURE_KEYS,
    DEFAULT_CONFIG,
    load_feature_manifest,
    results_to_frame,
    run_feature_pipeline,
    save_feature_table,
    summarise_feature_table,
)

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'processed_v4_rgb248_r4_exact' / 'manifest.csv'
MAX_FILES_ENV = os.getenv('FEATURE_EXTRACT_MAX_FILES', '').strip()
MAX_FILES = None if not MAX_FILES_ENV else int(MAX_FILES_ENV)
WORKERS = int(os.getenv('FEATURE_EXTRACT_WORKERS', str(min(8, os.cpu_count() or 4))))
FORCE_RERUN = os.getenv('FEATURE_EXTRACT_FORCE_RERUN', '0') == '1'
SHOW_PROGRESS = os.getenv('FEATURE_EXTRACT_SHOW_PROGRESS', '0') == '1'
RUN_NAME = 'feature_extraction_v2_rgb248_exact' if MAX_FILES is None else f'feature_extraction_v2_rgb248_exact_smoke_{MAX_FILES}'
OUTPUT_CSV = PROJECT_ROOT / 'features' / (f'{RUN_NAME}.csv')
AUDIT_ROOT = PROJECT_ROOT / 'audit_output' / 'validation' / RUN_NAME
SUMMARY_PATH = AUDIT_ROOT / 'feature_extraction_summary.json'
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DEFAULT_CONFIG

print({'manifest': str(MANIFEST_PATH), 'max_files': MAX_FILES, 'workers': WORKERS, 'output_csv': str(OUTPUT_CSV), 'feature_version': CONFIG.feature_version})

{'manifest': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact\\manifest.csv', 'max_files': None, 'workers': 8, 'output_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv', 'feature_version': 'v2_rgb248_exact_multibranch'}


## 1. Load accepted preprocessing manifest

Cell này chỉ đọc manifest preprocessing v4, lọc `ACCEPTED`, gán `split_role`, và tùy chọn lấy sample smoke theo `FEATURE_EXTRACT_MAX_FILES`.

In [2]:
manifest = load_feature_manifest(MANIFEST_PATH, config=CONFIG, max_files=MAX_FILES)
manifest[['generator', 'label', 'split_role', 'patch_path']].head(10)

,generator,label,split_role,patch_path
0,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
1,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
2,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
3,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
4,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
5,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
6,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
7,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
8,ADM,ai,val,C:\Users\USER\Desktop\ai_detector_img\data\pro...
9,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...


## 2. Run or load feature extraction

Nếu file output đã tồn tại và `FORCE_RERUN=False`, notebook sẽ load lại. Nếu không, notebook sẽ chạy full extraction bằng API package.

In [3]:
if OUTPUT_CSV.exists() and not FORCE_RERUN:
    feature_frame = pd.read_csv(OUTPUT_CSV)
else:
    results = run_feature_pipeline(
        manifest,
        config=CONFIG,
        workers=WORKERS,
        chunksize=32,
        show_progress=SHOW_PROGRESS,
    )
    feature_frame = results_to_frame(results, config=CONFIG)
    save_feature_table(feature_frame, OUTPUT_CSV)
summary = summarise_feature_table(feature_frame, config=CONFIG)
summary.update({'run_name': RUN_NAME, 'output_csv': str(OUTPUT_CSV), 'max_files': MAX_FILES, 'workers': WORKERS})
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
summary

Falling back to workers=1 because ProcessPoolExecutor is not safe from the current entrypoint.


{'rows': 85615,
 'ok_rows': 85615,
 'error_rows': 0,
 'feature_count': 36,
 'split_role_counts': {'train_core': 44235,
  'ood_eval': 27410,
  'val': 5821,
  'id_test': 5821,
  'calibration': 2328},
 'generator_counts': {'SDv15': 15670,
  'GLIDE': 11740,
  'SDv14': 11729,
  'Wukong': 11727,
  'ADM': 11721,
  'VQDM': 11717,
  'Midjourney': 11311},
 'cfa_validity_score': {'mean': -0.7266283469426945,
  'std': 0.2809198742432796,
  'q10': -1.0514668111815142,
  'q50': -0.7193991533185553,
  'q90': -0.3838814007841005},
 'run_name': 'feature_extraction_v2_rgb248_exact',
 'output_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv',
 'max_files': None,
 'workers': 8}

## 3. Status and split QA

Kiểm tra nhanh trạng thái extraction, số hàng theo split, và shape output.

In [4]:
feature_frame.groupby(['split_role', 'status']).size().unstack(fill_value=0)

status,ok
split_role,
calibration,2328
id_test,5821
ood_eval,27410
train_core,44235
val,5821


## 4. Feature preview

Xem một số cột quan trọng của nhánh `always-on` và `conditional`.

In [5]:
preview_cols = [
    'generator', 'label', 'split_role', 'status',
    'frs_mid_variance', 'fft_mid_logenergy', 'spatial_snr_ratio',
    'cfa_rg_pi_xy', 'cfa_bg_pi_xy', 'cfa_validity_score'
]
feature_frame[preview_cols].head(12)

,generator,label,split_role,status,frs_mid_variance,fft_mid_logenergy,spatial_snr_ratio,cfa_rg_pi_xy,cfa_bg_pi_xy,cfa_validity_score
0,ADM,ai,train_core,ok,0.887652,-0.393833,0.637813,0.005708,0.002024,-0.964877
1,ADM,ai,train_core,ok,0.734460,-0.542916,1.272391,0.000337,0.004975,-0.612072
2,ADM,ai,train_core,ok,0.597331,-0.565458,1.237888,0.001310,0.000855,-0.291873
3,ADM,ai,train_core,ok,0.145358,-0.285882,0.540529,0.001583,0.005174,-0.187073
4,ADM,ai,train_core,ok,0.823451,-0.426653,0.687906,0.005091,0.001253,-0.899774
5,ADM,ai,train_core,ok,0.507895,-0.345822,1.246961,0.010666,0.000143,-0.780397
6,ADM,ai,train_core,ok,0.277732,-0.171418,0.396057,0.002343,0.000596,-0.401830
7,ADM,ai,train_core,ok,0.384518,-0.273681,0.455329,0.006556,0.009937,-0.413620
8,ADM,ai,val,ok,0.641792,-0.447203,0.698859,0.000099,0.004693,-0.793823
9,ADM,ai,train_core,ok,0.568079,-0.499708,1.141728,0.002649,0.005715,-0.436104


## 5. Conditional CFA validity summary

Cell này chỉ xem phân bố `cfa_validity_score` để phục vụ bước audit/gating phía sau.

In [6]:
feature_frame['cfa_validity_score'].describe()

count    85615.000000
mean        -0.726628
std          0.280922
min         -2.755264
25%         -0.892523
50%         -0.719399
75%         -0.553889
max          0.507436
Name: cfa_validity_score, dtype: float64